### 导入数据集

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F

### 搭建网络结构

#### 使用MNIST训练模型

In [3]:
# 下载并加载MNIST训练数据集：28*28*1的灰度图像
train_dataset_mnist = datasets.MNIST(
    root='./data',
    train=True,
    transform=transforms.ToTensor(), # 转换为张量,缩放的范围为[0,1]
    download=True
)

# 创建训练集数据加载器
train_loader_mnist = DataLoader(
    dataset=train_dataset_mnist,
    batch_size=64,
    shuffle=True
)

In [4]:
# 定义神经网络结构
class Net(torch.nn.Module):  # 修复: Models -> Module
    def __init__(self):
        super(Net, self).__init__()
        self.L1 = torch.nn.Linear(784, 512) # 输入28*28=784,隐藏层512个神经元
        self.L2 = torch.nn.Linear(512, 256)
        self.L3 = torch.nn.Linear(256, 128)
        self.L4 = torch.nn.Linear(128, 64)
        self.L5 = torch.nn.Linear(64, 10) # 输出10类

    def forward(self, x):  # 修复: forword -> forward
        x = x.view(-1, 28*28) # 展平
        x = F.relu(self.L1(x))
        x = F.relu(self.L2(x))
        x = F.relu(self.L3(x))
        x = F.relu(self.L4(x))
        x = self.L5(x)
        return x 

# 创建网络实例并迁移到GPU(如果可用)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)

# 定义损失函数和优化器
loss_fn = torch.nn.CrossEntropyLoss() # 交叉熵损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam优化器

# 训练模型
def train(epoch):
    running_loss = 0.0  # 修复: runing_loss -> running_loss
    for batch_idx, data in enumerate(train_loader_mnist, 0):  # 修复: train_dataset -> train_loader
        inputs, target = data
        # 将数据移动到设备上 (CPU或GPU)
        inputs, target = inputs.to(device), target.to(device)
        
        optimizer.zero_grad() # 梯度清零
        outputs = model(inputs) # 修复: model.forword -> model (直接调用forward)
        loss = loss_fn(outputs, target) # 计算损失
        loss.backward() # 反向传播
        optimizer.step()  # 添加: 参数更新步骤

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print(f'[{epoch + 1}, {batch_idx + 1:5d}] loss: {running_loss / 300:.3f}')
            running_loss = 0.0
        
if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
    print("训练完成")

[1,   300] loss: 0.536
[1,   600] loss: 0.192
[1,   900] loss: 0.156
[2,   300] loss: 0.111
[2,   600] loss: 0.104
[2,   900] loss: 0.098
[3,   300] loss: 0.068
[3,   600] loss: 0.071
[3,   900] loss: 0.076
[4,   300] loss: 0.051
[4,   600] loss: 0.056
[4,   900] loss: 0.055
[5,   300] loss: 0.036
[5,   600] loss: 0.045
[5,   900] loss: 0.046
[6,   300] loss: 0.036
[6,   600] loss: 0.035
[6,   900] loss: 0.035
[7,   300] loss: 0.028
[7,   600] loss: 0.033
[7,   900] loss: 0.030
[8,   300] loss: 0.023
[8,   600] loss: 0.025
[8,   900] loss: 0.030
[9,   300] loss: 0.017
[9,   600] loss: 0.023
[9,   900] loss: 0.029
[10,   300] loss: 0.018
[10,   600] loss: 0.020
[10,   900] loss: 0.022
训练完成


#### 测试模型

In [5]:
# 加载测试数据集
test_dataset_mnist = datasets.MNIST(
    root='./data',
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

# 创建测试集数据加载器
test_loader_mnist = DataLoader(
    dataset=test_dataset_mnist,
    batch_size=64,
    shuffle=False
)

# 测试模型
def test_mnist():
    model.eval()  # 设置模型为评估模式
    correct = 0
    total = 0
    
    with torch.no_grad():  # 禁用梯度计算，节省内存和计算时间
        for data in test_loader_mnist:
            images, labels = data
            # 将数据移动到设备上
            images, labels = images.to(device), labels.to(device)
            
            # 前向传播
            outputs = model(images)
            
            # 获取预测结果
            _, predicted = torch.max(outputs.data, 1)
            
            # 统计总数和正确预测数
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    # 计算准确率
    accuracy = 100 * correct / total
    print(f'MNIST测试集准确率: {accuracy:.2f}% ({correct}/{total})')
    return accuracy

# 运行测试
if __name__ == '__main__':
    print("开始测试MNIST模型...")
    test_mnist()

开始测试MNIST模型...
MNIST测试集准确率: 98.16% (9816/10000)


## 搭建网络结构
### 使用CIFAR

In [6]:
# 下载并加载CIFAR-10数据集：32*32*3的彩色图像
cifar_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)
# 创建数据加载器
train_loader_cifar = DataLoader(
    dataset=cifar_dataset,
    batch_size=64,
    shuffle=True
)

In [7]:
# 定义适用于CIFAR-10的神经网络结构
class CifarNet(torch.nn.Module):
    def __init__(self):
        super(CifarNet, self).__init__()
        # CIFAR-10图像是32x32x3，输入特征数为32*32*3=3072
        self.L1 = torch.nn.Linear(3072, 1024)  # 输入层到第一隐藏层
        self.L2 = torch.nn.Linear(1024, 512)   # 第一隐藏层到第二隐藏层
        self.L3 = torch.nn.Linear(512, 256)    # 第二隐藏层到第三隐藏层
        self.L4 = torch.nn.Linear(256, 128)    # 第三隐藏层到第四隐藏层
        self.L5 = torch.nn.Linear(128, 10)     # 第四隐藏层到输出层（10类）

    def forward(self, x):
        x = x.view(-1, 32*32*3)  # 展平CIFAR图像：32*32*3=3072
        x = F.relu(self.L1(x))
        x = F.relu(self.L2(x))
        x = F.relu(self.L3(x))
        x = F.relu(self.L4(x))
        x = self.L5(x)
        return x

# 创建CIFAR网络实例并迁移到GPU(如果可用)
cifar_model = CifarNet().to(device)

# 定义损失函数和优化器
cifar_loss_fn = torch.nn.CrossEntropyLoss()
cifar_optimizer = torch.optim.Adam(cifar_model.parameters(), lr=0.01)

# 训练CIFAR模型
def train_cifar(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader_cifar, 0):
        inputs, target = data
        # 将数据移动到设备上 (CPU或GPU)
        inputs, target = inputs.to(device), target.to(device)
        
        cifar_optimizer.zero_grad()  # 梯度清零
        outputs = cifar_model(inputs)  # 前向传播
        loss = cifar_loss_fn(outputs, target)  # 计算损失
        loss.backward()  # 反向传播
        cifar_optimizer.step()  # 参数更新

        running_loss += loss.item()
        if batch_idx % 200 == 199:  # 每200个批次打印一次损失
            print(f'CIFAR-10 [{epoch + 1}, {batch_idx + 1:5d}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0

# 开始训练CIFAR模型
if __name__ == '__main__':
    print("开始训练CIFAR-10模型...")
    for epoch in range(10):  # 训练10个epoch
        train_cifar(epoch)
    print("CIFAR-10模型训练完成")

开始训练CIFAR-10模型...
CIFAR-10 [1,   200] loss: 2.546
CIFAR-10 [1,   400] loss: 2.079
CIFAR-10 [1,   600] loss: 2.068
CIFAR-10 [2,   200] loss: 2.054
CIFAR-10 [2,   400] loss: 2.081
CIFAR-10 [2,   600] loss: 2.073
CIFAR-10 [3,   200] loss: 2.075
CIFAR-10 [3,   400] loss: 2.055
CIFAR-10 [3,   600] loss: 2.045
CIFAR-10 [4,   200] loss: 2.035
CIFAR-10 [4,   400] loss: 2.037
CIFAR-10 [4,   600] loss: 2.047
CIFAR-10 [5,   200] loss: 2.037
CIFAR-10 [5,   400] loss: 2.040
CIFAR-10 [5,   600] loss: 2.037
CIFAR-10 [6,   200] loss: 2.040
CIFAR-10 [6,   400] loss: 2.036
CIFAR-10 [6,   600] loss: 2.030
CIFAR-10 [7,   200] loss: 2.042
CIFAR-10 [7,   400] loss: 2.043
CIFAR-10 [7,   600] loss: 2.032
CIFAR-10 [8,   200] loss: 2.038
CIFAR-10 [8,   400] loss: 2.036
CIFAR-10 [8,   600] loss: 2.027
CIFAR-10 [9,   200] loss: 2.040
CIFAR-10 [9,   400] loss: 2.023
CIFAR-10 [9,   600] loss: 2.034
CIFAR-10 [10,   200] loss: 2.039
CIFAR-10 [10,   400] loss: 2.026
CIFAR-10 [10,   600] loss: 2.027
CIFAR-10模型训练完成


In [ ]:
# 加载测试数据集 - 修复：正确命名变量
test_dataset_cifar = datasets.CIFAR10(
    root='./data',
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

# 创建测试集数据加载器 - 修复：使用正确的变量名
test_loader_cifar = DataLoader(
    dataset=test_dataset_cifar,
    batch_size=64,
    shuffle=False
)

# 测试模型
def test_cifar():
    cifar_model.eval()  # 修复：使用正确的模型(cifar_model而不是model)
    correct = 0
    total = 0
    
    with torch.no_grad():  # 禁用梯度计算，节省内存和计算时间
        for data in test_loader_cifar:  # 修复：使用正确的数据加载器
            images, labels = data
            # 将数据移动到设备上
            images, labels = images.to(device), labels.to(device)
            
            # 前向传播 - 修复：使用正确的模型
            outputs = cifar_model(images)
            
            # 获取预测结果
            _, predicted = torch.max(outputs.data, 1)
            
            # 统计总数和正确预测数
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    # 计算准确率
    accuracy = 100 * correct / total
    print(f'CIFAR-10测试集准确率: {accuracy:.2f}% ({correct}/{total})')
    return accuracy

# 运行测试
if __name__ == '__main__':
    print("开始测试CIFAR-10模型...")
    test_cifar()

开始测试CIFAR10模型...


RuntimeError: shape '[-1, 784]' is invalid for input of size 196608